# 08_04 LoRA: fine-tuning with far fewer numbers

*📖 Book: appendix E (parameter-efficient fine-tuning with LoRA)*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: the adult takes notes on a small notepad
>
> Project files this connects to:
>
> - `src/homer_ai/lora.py`
> - `src/homer_ai/finetune.py`
> - `src/homer_ai/gpt.py`
>
> **This notebook:** Fine-tuning changes millions of weights, and every job needs its own changed copy. LoRA trains two thin tables beside each layer and leaves the model itself alone.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What a low-rank table is, with a 4 × 4 example you can check by hand
- How to write a LoRA layer in about 20 lines
- How many numbers LoRA trains, compared with 08_02
- What LoRA gives (small files, an untouched model) and what it costs on a CPU (time)

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **LoRA** | low-rank adaptation: the model's weights stay frozen, and beside each chosen layer two thin tables are trained, whose product is added to the layer's result | `W` frozen, `B @ A` trained |
| **rank** | here: the width of the two thin tables. A bigger rank can express a bigger change and costs more numbers | rank 8 |
| **adapter** | the small set of trained numbers that turns a frozen base model into a model for one job | a 5.4 MB file next to GPT-2's 548 MB |

Words you've met before: **merge** ([02_03](../02_text_to_numbers/02_03_bpe.ipynb)).

## 3. The idea

### ❓ What problem does LoRA solve?

In [08_02](08_02_spam_classifier.ipynb) we trained 7 million numbers, and the result was a changed last block: 28 MB that belong to the spam job only. In [08_03](08_03_instructions.ipynb) we trained all 163 million. Now think of a company with 50 jobs for one model. That's 50 changed copies. With today's models, each copy is many gigabytes.

**LoRA** (low-rank adaptation) leaves every weight of the model exactly as it is. Beside a layer it puts two thin tables, and trains only those.

In the picture of [08_00](08_00_chapter_overview.ipynb): the adult takes notes on a small notepad instead of rewiring their brain. One brain, and a notepad per job.

### ❓ How can two thin tables stand in for a big one?

Fine-tuning a layer changes its weight table `W` into `W + ΔW`, where `ΔW` is the change, a table of the same shape. For a 768 × 768 layer, that's 589,824 numbers to learn.

LoRA's bet is that the change is simple enough to be written as a product of two thin tables:

```
ΔW  =  B @ A          B: (768, r)     A: (r, 768)      r is the rank, a small number
```

Here it is small enough to do by hand, with 4 in place of 768 and rank 1:

```
B (4 × 1)      A (1 × 4)                 B @ A (4 × 4)
  1                                        1  0  2  0
  2        @   1  0  2  0       =          2  0  4  0
  0                                        0  0  0  0
  3                                        3  0  6  0
```

Eight numbers made a table of sixteen. Every row of the result is the row `A`, times one number from `B`. So it can't be *any* 4 × 4 table, only a simple, repetitive one. That's what **low rank** means.

👉 **Predict first:** for 768 × 768 and rank 8, how many numbers are in `A` and `B` together?

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: is the hand example right, and what about 768?

`@` is matrix multiplication ([A_03](../A_toolbox/A_03_dot_product.ipynb)).

In [ ]:
import torch

B = torch.tensor([[1.0], [2.0], [0.0], [3.0]])  # (4, 1)
A = torch.tensor([[1.0, 0.0, 2.0, 0.0]])  # (1, 4)
print(B @ A)
print("numbers in A and B:", A.numel() + B.numel())
print("numbers in the big table:", (B @ A).numel())
print()
for rank in (1, 8, 64):
    thin = 2 * rank * 768
    share = thin / 768**2
    print(f"768 x 768 at rank {rank:2}: {thin:>7,} numbers, {share:6.1%} of 589,824")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The same 4 × 4 table as above, from 8 numbers. For a real 768 × 768 layer, rank 8 needs 12,288 numbers: 2.1% of the full table.

</details>

The layer we're about to write, as a picture:

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    GOOD,
    INK,
    MUTED,
    arrow,
    box,
    diagram_axes,
    plus,
    studio_figure,
    tag,
)

fig = studio_figure("LoRA", "A frozen table, and two thin ones beside it", (10, 6.4))
ax = diagram_axes(fig)

box(ax, 0.9, 2.9, 1.2, 0.9, "x\n768", INK, mono=True)
box(ax, 5.0, 4.3, 3.0, 1.5, "W\n768 × 768", MUTED, mono=True, fontsize=12, dashed=True)
tag(ax, 5.0, 5.35, "frozen: 589,824 numbers, never nudged", MUTED, 9.5)
box(ax, 3.9, 1.5, 1.9, 0.5, "A  8 × 768", GOOD, mono=True, lw=2.2)
box(ax, 6.3, 1.5, 0.6, 1.5, "B", GOOD, mono=True, lw=2.2)
tag(ax, 6.3, 0.52, "768 × 8", GOOD, 9.5, mono=True)
tag(ax, 3.9, 0.95, "starts random", GOOD, 9)
tag(ax, 6.3, 0.2, "starts at zero", GOOD, 9)
tag(ax, 5.0, 2.6, "trained: 2 × 8 × 768 = 12,288 numbers", GOOD, 9.5)
arrow(ax, (1.5, 3.1), (3.5, 4.2), color=MUTED)
arrow(ax, (1.5, 2.7), (2.95, 1.6), color=GOOD)
arrow(ax, (4.85, 1.5), (6.0, 1.5), "8", color=GOOD, mono=True)
plus(ax, 8.1, 2.9)
arrow(ax, (6.5, 4.2), (7.9, 3.1), color=MUTED)
arrow(ax, (6.6, 1.6), (7.9, 2.7), color=GOOD)
arrow(ax, (8.35, 2.9), (8.9, 2.9), color=ACCENT)
box(ax, 9.45, 2.9, 1.0, 0.9, "out\n768", ACCENT, mono=True)
plt.show()

### ❓ Step 2: what does a LoRA layer look like in code?

It wraps an existing `nn.Linear`. The old layer is frozen. `A` and `B` are new, and their result is added to the old layer's result.

```
out = linear(x)  +  scale * (x @ A.T @ B.T)
      └ frozen ┘    └──── trained ────────┘
```

Two details matter:

- **`B` starts at zero.** Then `B @ A` is all zeros, and the wrapped layer gives exactly what the old layer gave. Fine-tuning starts from GPT-2 as it is, not from a disturbed copy. (`A` starts with small random numbers. If both were zero, neither could ever get a nudge.)
- **`scale = alpha / rank`** sets how strongly the change counts. A common habit is `alpha = 2 × rank`, so scale 2.

In [ ]:
import math

from torch import nn


class LoRALinear(nn.Module):
    def __init__(self, linear, rank=8, alpha=16.0):
        super().__init__()
        self.linear = linear
        for parameter in linear.parameters():
            parameter.requires_grad = False  # frozen: the optimizer leaves it alone
        self.A = nn.Parameter(torch.empty(rank, linear.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))  # small random numbers
        self.B = nn.Parameter(torch.zeros(linear.out_features, rank))
        self.scale = alpha / rank

    def forward(self, x):
        return self.linear(x) + self.scale * (x @ self.A.T @ self.B.T)


torch.manual_seed(0)
plain = nn.Linear(768, 768)
wrapped = LoRALinear(plain, rank=8)
x = torch.randn(2, 768)
print("same answer as the plain layer at the start:", torch.equal(wrapped(x), plain(x)))
for name, parameter in wrapped.named_parameters():
    state = "trained" if parameter.requires_grad else "frozen"
    shape = str(tuple(parameter.shape))
    print(f"{name:14} {shape:12} {parameter.numel():>8,}  {state}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

At the start the wrapped layer answers exactly like the plain one. Inside it, the 768 × 768 weight and its bias are frozen, and `A` (8 × 768) and `B` (768 × 8) are trained: 6,144 numbers each.

`nn.Parameter` is how you tell PyTorch "this tensor is a weight of my layer", so that the optimizer finds it. The same class is in `lora.py`, with a `merged()` method we use at the end.

</details>

### ❓ Step 3: how do we put it into GPT-2?

Same start as [08_02](08_02_spam_classifier.ipynb): the spam data, GPT-2 in our class, a new 2-score head. But this time **every** GPT-2 weight is frozen, the last block too. Only the new head is trained in full, because it starts random and is tiny.

Then `apply_lora` walks through the model and wraps every `nn.Linear` it finds inside the blocks: query, key, value, the output projection and the two feed-forward layers. That's 6 layers in each of 12 blocks.

👉 **Predict first:** 08_02 trained 7,090,946 numbers. Will this be more or fewer, with adapters in all 12 blocks?

In [ ]:
import tiktoken

from homer_ai.finetune import balance, encode_messages, load_sms_spam, split

# the data of 08_02, prepared the same way: balanced, split, padded to 64 tokens
encoding = tiktoken.get_encoding("gpt2")
train, val, test = split(balance(load_sms_spam(), seed=123))
train_ids, train_labels = encode_messages(train, encoding, 64)
val_ids, val_labels = encode_messages(val, encoding, 64)
test_ids, test_labels = encode_messages(test, encoding, 64)
print(len(train), "training,", len(val), "validation,", len(test), "test messages")

In [ ]:
import tiktoken
import torch

from homer_ai.pretrained import load_gpt2

encoding = tiktoken.get_encoding("gpt2")
model = load_gpt2()  # GPT-2 small inside our own GPTModel (08_01); read from the cache

In [ ]:
from homer_ai.finetune import accuracy, make_classifier, train_classifier
from homer_ai.lora import apply_lora, count_parameters

make_classifier(model)  # the new head, as in 08_02 ...
for parameter in model.parameters():
    parameter.requires_grad = False  # ... but now freeze ALL of GPT-2
for parameter in model.output_head.parameters():
    parameter.requires_grad = True  # the head is trained in full

wrapped_layers = apply_lora(model, rank=8, alpha=16, skip=("output_head",))
trainable, total = count_parameters(model)
print("layers wrapped:", wrapped_layers)
print(f"trained:        {trainable:>11,}")
print(f"all:            {total:>11,}")
print(f"share:          {trainable / total:.1%}")
print()
print(model.blocks[0].attention.query)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

72 layers wrapped, and 1,328,642 trained numbers: about 1.1% of the model, and 5 times fewer than in 08_02, even though now all 12 blocks can adapt and not only the last. (The total grew a little, to 125,768,450, because the thin tables were added.)

The last lines show one wrapped layer: a `LoRALinear` with the old `Linear` inside.

</details>

### ❓ Step 4: does it learn the spam job?

Same data, same loop, same batch size as 08_02. Two things differ. The learning rate is higher, 0.0003 against 0.00005, because `B` starts at zero and has a long way to go. And we train 2 epochs, not 3, because each epoch is slower here (the next question says why).

👉 **Predict first:** 08_02 reached 93.7% on the test set. Will 5 times fewer numbers do worse?

In [ ]:
import time

started = time.time()
train_classifier(model, train_ids, train_labels, epochs=2, lr=3e-4, report_every=30)
train_seconds = time.time() - started
print(f"training took {train_seconds:.0f} seconds")

In [ ]:
results = {
    "training (first 300)": accuracy(model, train_ids[:300], train_labels[:300]),
    "validation": accuracy(model, val_ids, val_labels),
    "test": accuracy(model, test_ids, test_labels),
}
for name, value in results.items():
    print(f"{name:22} {value:.1%}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The loss falls as in 08_02, and ends near 0.2. On my laptop the two epochs took about 117 seconds, and the accuracies are: training 97.3%, validation 95.3%, test **94.3%**.

So no, not worse. With a fifth of the trained numbers it lands where 08_02 did.

</details>

### ❓ So which is better, 08_02 or LoRA?

Here are both runs from my laptop, side by side:

| | 08_02: last block trained | 08_04: LoRA, rank 8 |
|---|---|---|
| GPT-2's own weights | last block changed | all untouched |
| Trained numbers | 7,090,946 | 1,328,642 |
| Epochs | 3 | 2 |
| Training time | about 65 s | about 117 s |
| Time per step | 0.17 s | 0.45 s |
| Validation accuracy | 94.6% | 95.3% |
| Test accuracy | 93.7% | 94.3% |
| Size of what was trained | 28 MB | 5.4 MB |

The accuracy is about the same. LoRA trains 5 times fewer numbers, and its result is a much smaller file. But each step took more than twice as long, and I'd rather explain that than hide it.

In 08_02 only the last block was trained, so working out the nudges could stop after one block. LoRA has adapters in all 12 blocks, so the nudges have to be worked out all the way down. Fewer numbers to store doesn't mean less work per step.

So on a laptop, for one job, freezing all but the last block is the better deal. LoRA pays off in other ways:

- **Memory.** Every trained weight needs room for its nudge and for the optimizer's notes about it. On big models that's what fills the graphics card. Fewer trained weights, less memory.
- **Files.** One base model, and a small adapter per job.
- **Safety of the base.** GPT-2's weights are untouched, so nothing it knew can be lost. Remove the adapter and you have the original back.

### ❓ Step 5: how small is the file?

We save only what was trained: the `A` and `B` tables and the head. It goes to the cache folder outside the project, next to GPT-2's file. (Fine-tuned models don't belong in git. Even small ones add up.)

In [ ]:
from homer_ai.pretrained import CACHE, GPT2_FILE

adapter = {name: p.detach() for name, p in model.named_parameters() if p.requires_grad}
adapter_file = CACHE / "spam_lora_adapter.pt"
torch.save(adapter, adapter_file)

print(adapter_file)
print(f"adapter: {adapter_file.stat().st_size / 1e6:6.1f} MB  ({len(adapter)} tensors)")
print(f"GPT-2:   {GPT2_FILE.stat().st_size / 1e6:6.1f} MB")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

5.4 MB for the adapter, next to 548 MB for GPT-2: about 1%. To use it elsewhere you'd load GPT-2, apply LoRA the same way, and load these 146 tensors on top.

</details>

### ❓ Step 6: does the notepad stay forever?

It doesn't have to. Once training is done, `B @ A` is just a table of numbers of the same shape as `W`. Add it in, and the layer is one plain `nn.Linear` again, with no extra work at run time. That's called **merging**.

```
W_merged = W + scale * (B @ A)
```

`merged()` in `lora.py` does that for one layer. Let's check that the merged layer answers like the wrapped one.

In [ ]:
layer = model.blocks[11].attention.query  # a LoRALinear, after training
merged = layer.merged()  # a plain nn.Linear

x = torch.randn(4, 768)
with torch.no_grad():
    difference = (merged(x) - layer(x)).abs().max().item()
    change = layer.scale * (layer.B @ layer.A)
print("merged layer:", merged)
print(f"biggest difference in the answers: {difference:.6f}")
print(f"typical size of a weight in W:     {layer.linear.weight.abs().mean():.4f}")
print(f"typical size of the change B @ A:  {change.abs().mean():.4f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The merged layer is a plain `Linear(768 → 768)` and its answers match the wrapped layer's to about five decimal places (the tiny rest is rounding). And look at the sizes: a typical weight in `W` is about 0.087, and a typical entry of the change is about 0.0005. The notepad holds a light correction, not a new brain.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ How do I choose the rank?

Start small. Ranks of 4 to 16 are common. A higher rank can express a bigger change, costs more numbers, and on a small data set like ours rarely helps. Change `rank=8` above to 2 or 32 and compare the validation accuracy.

### ❓ Why wrap all six layers of a block?

That's what Raschka's book does, and it's the simple choice. The paper that introduced LoRA wrapped only query and value, which trains fewer numbers still. Which layers matter most depends on the job.

### ❓ Is the comparison with 08_02 fair?

Mostly. Same data, same split, same seed, same batch size, same test set. Two things differ besides the method: the learning rate (each method got one that suits it) and the number of epochs (3 against 2). So read the table as "both get to about 94%", not as proof that LoRA is better. On 300 test messages, the 0.6 points between them is two messages.

### ❓ Could I use LoRA for the instruction job of 08_03?

Yes, the same `apply_lora` works there (without the `skip`, since there's no new head). It would train about 1.3 million numbers and not 163 million. Don't expect it to fix the weak results of 08_03, though. Those come from the model's size and the short training, not from the method.

## 5. Recap

- LoRA freezes the model and trains two thin tables beside each layer. Their product, `B @ A`, is a low-rank change added to the layer's result.
- `B` starts at zero, so fine-tuning starts from the model exactly as it was.
- On the spam job LoRA trained 1.3 million numbers against 7.1 million in 08_02, and reached about the same accuracy.
- On a CPU it was slower, because the nudges are worked out through all 12 blocks. Its gains are memory, small adapter files, and a base model that stays untouched.
- After training, the change can be merged into the weights, so the layer is plain again.

**Next:** [09_00: Answers grounded in Homer's text](../09_advanced_grounded_answers/09_00_chapter_overview.ipynb). Or go [back to the course map](../00_start_here/00_00_course_map.ipynb).